# The Agent Loop

**Prerequisites:** `01_workflows_vs_agents.ipynb`, and from `01_foundations`: `06_function_calling.ipynb`, `09_build_an_agent.ipynb`

`asend_with_tools()` (chapter 1, notebook 9) is already an agent loop: call the model, run whatever tool it asks for, feed the result back, go around again. It works. What it can't do is change shape — the order of operations *is* the function body, so every agent it can ever run is the same agent. Wanting a planner instead, or a critic at the end, means writing a different function.

This notebook takes that one function apart into four pieces that can be re-ordered, then puts them back together as the same ReAct loop — this time declared as a list instead of hardcoded as control flow.

**By the end you'll have:**
- Built `Runtime`: a loop that walks a declared list of components, passing one `Conversation` between them, without ever inspecting what any of them produced
- Built `Act` (executes a decision already made) and `Observe` (turns a raw effect into something the conversation can read), and seen why a tool's raw output deliberately never lands in `messages`
- Built `Think`, and seen why translating our own roles back into the API's takes real work: flattening them to prose makes the model re-request a tool it has already run
- Run ReAct end to end as a declared loop, then made that loop safe to leave running: four budgets, doom-loop detection, and a graceful stop that says what actually happened

## Setup

Same pattern as `01_foundations`: `runtime.py`, `observe.py`, `think.py`, and `act.py` live one directory up, and they in turn import `llm.py` and `chat.py` from `01_foundations`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think
from act import Act

## What one function was doing at once

Here is chapter 1's agent loop again, unchanged:

```python
async def asend_with_tools(self, content, tools, functions, schemas=None, retries=3, model=DEFAULT_MODEL):
    self.messages.append({"role": "user", "content": content})
    for _ in range(retries):
        response = await async_client.chat.completions.create(model=model, messages=self.messages, tools=tools)
        message = response.choices[0].message
        self.messages.append(message)

        if not message.tool_calls:
            return message.content

        results = await asyncio.gather(*(run(tc) for tc in message.tool_calls))
        for tool_call, result in zip(message.tool_calls, results):
            self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
```

Three different jobs are braided together in those lines:

1. **Deciding** — `client.chat.completions.create(...)` looks at the history and picks a tool call or a final answer.
2. **Executing** — `run(tc)` actually calls the Python function and changes something outside the program.
3. **Observing** — the `self.messages.append({"role": "tool", ...})` line takes what came back and puts it somewhere the next decision can see it.

And one job is doing all the coordinating: the `for _ in range(retries)` loop itself, which decides that step 2 follows step 1 and that everything repeats until `not message.tool_calls`.

Those are the four pieces. Chapter 1 fused them because a fused version is shorter and, for one fixed pattern, perfectly good. Separating them buys exactly one thing — the order stops being control flow and becomes data — and that one thing is what the rest of this chapter is built on.

| chapter 1, inside `asend_with_tools()` | this chapter |
|---|---|
| `for _ in range(retries)` | `Runtime` — walks a declared list, `while not finished` |
| `chat.completions.create(...)` | `Think` — the only component allowed to reason |
| `run(tool_call)` / `execute_tool_call` | `Act` — executes an already-made decision |
| `messages.append({"role": "tool", ...})` | `Observe` — turns a raw effect into a readable observation |

The same two tools as chapter 1's notebook 9, unchanged — real functions, real Pydantic schemas, one of which has a custom validator that rejects a timezone that doesn't exist:

In [2]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

def get_current_time(timezone):
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_current_time": get_current_time, "convert_temperature": convert_temperature}
SCHEMAS = {"get_current_time": GetCurrentTimeArgs, "convert_temperature": ConvertTemperatureArgs}
tools = [
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {
            "type": "object",
            "properties": {"timezone": {"type": "string", "description": "IANA timezone name, e.g. Asia/Tokyo"}},
            "required": ["timezone"],
        },
    }},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "from_unit": {"type": "string", "enum": ["C", "F"]},
                "to_unit": {"type": "string", "enum": ["C", "F"]},
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    }},
]

## `runtime.py`: the loop

The replacement for `for _ in range(retries)`. It holds a list of components, hands each one the same `Conversation`, and appends whatever comes back:

```python
class Runtime:
    def __init__(self, loop=None, repeat_from=None, ...):
        self.loop = list(loop) if loop else []
        self.repeat_from = repeat_from
        self._position = 0

    def add(self, component):
        self.loop.append(component)
        return self

    @property
    def finished(self):
        return self.repeat_from is None and self._position >= len(self.loop)

    def next(self):
        if self._position >= len(self.loop):
            self._position = self.repeat_from
        component = self.loop[self._position]
        self._position += 1
        return component

    def run(self, conversation):
        while not self.finished:
            component = self.next()
            output = component.run(conversation)
            if output is not None:
                conversation.messages.append(output)

            if conversation.messages and conversation.messages[-1].get("role") == "assistant":
                return conversation
        return conversation
```

Every component shares one interface — `run(conversation)` — and that is the entire contract. `runtime.py` never imports `Observe`, `Think`, or `Act`; it doesn't import `Conversation` either. It only knows that the things in `self.loop` have a `.run()`, and that `conversation` has a `.messages` list.

**`next()` decides what runs next, and that is not reasoning.** It reads `self._position`, an integer, and returns `self.loop[position]`. It never looks at what the last component produced. Compare that with `Think`, whose entire job is to look at content and judge it. Keeping this line honest is what lets one `Runtime` run every pattern in this chapter — the moment `next()` starts inspecting messages to pick a component, the pattern has been baked into the runtime and only one pattern can ever run.

**`repeat_from` is what makes it a loop rather than a pipeline.** Left as `None`, the runtime walks the list once and stops. Set to an index, walking off the end wraps back to that index instead of finishing. It's an index rather than a boolean specifically so a loop can have setup steps that run once: `repeat_from=1` re-runs everything except position 0.

**`if output is not None`** matters because not every component produces a message. `Act` running a tool changes the world but has nothing to say about it yet — it returns `None`. That gap is what `Observe` exists to fill.

**The `assistant` check is the only exit that isn't positional.** An `assistant` message means someone decided this is the final answer, so the run is over regardless of where in the list we are. It's still not reasoning: the runtime is reading a role string that a `Think` already set, not judging whether the answer is any good.

## `act.py`: executing a decision already made

`Act` never decides anything. It finds the most recent `decision` in the conversation and carries it out:

```python
class Act:
    def __init__(self, functions=None, schemas=None, retries=3, ...):
        self.functions = functions
        self.schemas = schemas

    def run(self, conversation):
        decision = pending_decision(conversation)
        if decision is None:
            return None
        decision["applied"] = True

        if decision["type"] == "respond":
            return {"role": "assistant", "content": decision["content"]}

        conversation.pending = self.execute(decision["tool"], decision["content"])
        return None

    def execute(self, tool_name, args):
        tool_call = SimpleNamespace(id="call", function=SimpleNamespace(name=tool_name, arguments=args))
        result = execute_tool_call(tool_call, self.functions, self.schemas)
        return {"tool": tool_name, "content": result}
```

`execute_tool_call()` comes straight from `01_foundations/06_function_calling.ipynb` — `Act` reuses chapter 1's tool execution rather than reimplementing it, along with its Pydantic validation of the arguments. (The real `execute()` also has a retry ladder around this; that's notebook 3.)

**`pending_decision` asks a more careful question than it looks.** The obvious version is
"find the most recent decision" — a single `next()` over the reversed messages. That was the
first version here, and it works only because of an arrangement nobody wrote down: every loop in
this chapter happens to put a `Think` immediately before every `Act`, so the newest decision is
always the fresh one.

Break that arrangement and it fails silently. Let a `Think` return something that *isn't* a
decision — a plan, a critique — and the next `Act` scans back past it, finds the decision from
the previous turn, and runs it a second time:

```python
def pending_decision(conversation):
    """The most recent decision nobody has carried out yet, or None."""
    return next((m for m in reversed(conversation.messages)
                 if m.get("role") == "decision" and not m.get("applied")), None)
```

`Act` stamps `applied` before carrying a decision out, so no later `Act` can find it again. It
was asking *"what is the latest decision?"* when what it meant was *"what is the latest decision
that has not happened yet"*. Returning `None` also matters: a turn where nothing was decided is
perfectly ordinary, not an error, and the earlier version raised `StopIteration` on it.

The `if decision["type"] == "respond"` branch is not `Act` exercising judgment. Some `Think` already wrote `"type": "respond"` into that message; `Act` is dispatching on a field it was handed, the same way `Runtime.next()` dispatches on an integer. Both are mechanical, and both are mechanical *because* the decision was already made somewhere it belonged.

**`decision` is a new role.** Chapter 1's conversations had `system`, `user`, `assistant`, and `tool` — the four the API defines. Splitting deciding from executing forces a fifth: something has to represent "a tool call has been chosen but not yet run." That distinction didn't exist in `asend_with_tools()`, where choosing and running were the same instant. This chapter adds `decision`, `plan`, and `critique` for the same reason. That has a cost, and paying it is the subject of the `Think` section below.

## `observe.py`: an effect is not an observation

Notice what `Act` did *not* do above: it never touched `conversation.messages`. It wrote the tool's result to `conversation.pending`, a plain attribute, and returned `None`.

That's deliberate. What a tool returns is raw — `'2026-08-07 06:54 JST'` is small, but a database query, an HTTP response, or a file read can be tens of thousands of tokens of JSON. Appending that straight into `messages` means every future `Think` re-reads all of it on every call, and chapter 1's notebook 3 already established what that costs: the context window is finite and everything in it is billed on every turn.

So the raw thing goes to `pending`, and `Observe` decides what the conversation actually gets to see:

```python
class Observe:
    def __init__(self, event=None, extractors=None):
        self.event = event
        self.extractors = extractors or {}

    def run(self, conversation):
        if self.event is not None:
            return {"role": "user", "content": self.event}

        effect = getattr(conversation, "pending", None)
        conversation.pending = None
        if effect is None:
            return None          # nothing happened, so there is nothing to observe

        extractor = self.extractors.get(effect["tool"])
        if extractor is not None:
            content = extractor(effect)
        else:
            content = chat([{"role": "user", "content": (
                f"Conversation so far:\n{render(conversation)}\n\n"
                f"The tool `{effect['tool']}` was just called and returned:\n{effect['content']}\n\n"
                "State this result as one plain factual sentence, using the conversation only "
                "to phrase it correctly (e.g. which units, which entity).\n\n"
                "Keep anything the TOOL itself said about what to do next -- a suggestion, a "
                "constraint, an alternative to try, the reason it refused. That is part of "
                "the result, and the agent reading this has to act on it.\n\n"
                "Add nothing of your own: no speculation about why the result matters, how it "
                "will be used, or what should happen now. No preamble, no commentary."
            )}])

        conversation.step_outputs.append(effect["content"])
        return {"role": "tool", "content": content}
```

Two modes, and the choice between them is a real engineering decision:

- **An `extractor`** is a plain Python function you supply per tool — `lambda e: json.loads(e["content"])["temp_c"]`. Deterministic, free, instant. Use it whenever you know the shape of what comes back.
- **No extractor** falls back to an LLM call that phrases the result in one sentence. Costs a call and can be wrong, but handles output whose shape you can't predict.

Either way, `Observe` reports; it doesn't decide. That line is the whole reason `Think` and `Observe` are separate components rather than one.

**But the fence is about *whose words* they are, not about what they say — and getting that wrong cost a run.** The first version of this prompt ended:

> Do not speculate about why it matters, **what happens next**, or how it will be used.

A weather tool then refused a lookup with:

> `No weather station for Vestby, Norway. Try a larger nearby city.`

and what reached the conversation was:

> `There is no weather station for Vestby, Norway.`

`Observe` did exactly as instructed. "Try a larger nearby city" is *what happens next*, so out it went — and the agent, now holding a dead end with no way forward, relayed the refusal to the user instead of retrying with a larger city. The tool had said precisely what to do; the observer deleted it.

Two different things were being confused: the observer's opinion, and the tool's own output. A tool's suggestion, constraint, alternative, or reason for refusing is **part of the result**. So the prompt now separates them — keep everything the tool said, add nothing of your own. `Observe` still cannot decide what to do; it just no longer destroys the evidence that would let `Think` decide.

This is also the strongest argument for `extractors`. A deterministic function returns exactly what the tool returned, so there is nothing to summarise and nothing to lose. Every result that goes through an LLM to be phrased is a result something can be dropped from.

**`if effect is None: return None`** is the counterpart to `Act` returning `None`. Some
decisions change the conversation rather than the world — notebook 4's replanning is the clear
case — and leave nothing in `pending`. Returning `None` means `Runtime` appends nothing, which
is the honest record: no effect, no observation.

`conversation.pending` is cleared the moment it's read: it's a handoff slot between two adjacent components, not state. And `conversation.step_outputs` keeps every raw result, indexed, outside `messages` — nothing needs it yet, but notebook 4 will, when a planned step has to reference an earlier step's exact output rather than a sentence about it.

**Two new non-message slots on `Conversation`.** Chapter 1's `Conversation` had `messages` and (from notebook 9) `notes`. `pending` and `step_outputs` follow the same principle `notes` established: not everything the agent knows belongs in what the model is shown.

`Observe(event=...)` is the other mode — bringing something in from outside rather than from a tool. It's how the user's request enters the loop.

## `think.py`: our roles are not the API's roles

`Think` is the only component allowed to reason, and the only one that talks to the model with `tools=`. That gives it a problem none of the others have: the conversation it has to read contains roles the API has never heard of.

The obvious translation is to flatten anything unrecognized into a role that is recognized — which is what the first version of this chapter did:

```python
messages = [
    {"role": m["role"] if m["role"] in ("system", "user", "assistant") else "user",
     "content": m["content"]}
    for m in conversation.messages
]
```

Every `decision` and every `tool` result becomes a `user` message. All the information is still there, in the right order, in plain text.

It doesn't work, and the way it fails is worth understanding: the model asks for `get_current_time`, is told the time in a plain sentence, and then asks for `get_current_time` again with identical arguments. It hasn't lost the answer — it just doesn't register that it already has one.

What flattening destroyed was *structure*, not information. Models are trained on a very specific shape for "I called a tool and here is what came back": an `assistant` message whose `content` is `null`, carrying a structured `tool_calls` array, followed by a `tool`-role message whose `tool_call_id` matches it. That pairing is the signal that the work is done. Rewritten as two `user` messages containing the same words, it reads as somebody narrating — and narration about a tool result is not the same signal as a tool result.

Chapter 1 never hit this, because it never had the problem. `asend_with_tools()` appended the SDK's `message` object directly, `tool_calls` and all, and let the API's own format pass through untouched. Inventing `decision` and `plan` roles is what broke that, so `Think` has to put it back.

### Rebuilding the shape the model was trained on

`_build_messages()` translates our conversation into a real one, reconstructing the tool-call pairing as it goes:

```python
def _build_messages(self, conversation):
    explain = (
        "\nBefore each tool call, briefly say why."
        "\nWhen you are answering instead of calling a tool, give the answer itself and "
        "nothing about your process."
    ) if self.explain else ""

    messages = [{
        "role": "system",
        "content": (
            "If the user's request has already been fully answered by a tool result shown "
            "below, respond in plain text with that answer -- do not call the tool again."
            + explain
        ),
    }]
    pending_call_id = None
    for m in conversation.messages:
        role = m["role"]
        if role in ("system", "user", "assistant"):
            messages.append({"role": role, "content": m["content"]})
        elif role == "decision" and m.get("type") == "tool_call":
            pending_call_id = f"call_{len(messages)}"
            messages.append({
                "role": "assistant",
                "content": m.get("reasoning"),
                "tool_calls": [{
                    "id": pending_call_id,
                    "type": "function",
                    "function": {"name": m["tool"], "arguments": m["content"]},
                }],
            })
        elif role == "tool" and pending_call_id is not None:
            messages.append({"role": "tool", "tool_call_id": pending_call_id, "content": m["content"]})
            pending_call_id = None
        elif role == "tool":
            messages.append({"role": "user", "content": f"Results:\n{m['content']}"})
        else:
            messages.append({"role": "assistant", "content": m["content"]})
    return messages
```

A `decision` of type `tool_call` becomes a real assistant tool-call message; the `tool` observation that follows it becomes a real tool reply carrying the same id. The ids are generated here rather than stored, because in our conversation a `tool_call` decision is always followed by exactly one `tool` observation — pairing by position is enough. Roles with no API equivalent (`plan`, `critique`) fall through to the `else` and are narrated as the agent's own prior turns, which is at least honest: they *were* the agent's own thinking.

**The second-to-last branch is there because of a bug that took a live run to find.** A `tool`
observation with no `tool_call` before it — a fork's merged branch results, or a search round's
survivors — used to fall through to that same `else` and be narrated as the agent's own turn.
Which put an **assistant** message last in the request. A model handed a trailing assistant
message does not reply to it; it *continues* it. Notebook 6's Tree of Thoughts answered:

> *" This approach seems preferable as it fosters a supportive environment, crucial for someone just starting out with running."*

A sentence fragment, picking up mid-thought from the pruning rationale above it, with a leading
space still attached. Every check passed — the run terminated, the last message had role
`assistant`, every component had run — and the answer was nonsense.

Results that arrive from elsewhere are things that happened *to* the agent, not things it said,
so they go in as `user` turns. Which is also what ends the turn and asks for a fresh reply.

### The system message, and why it is assembled rather than fixed

The first line is there because correct structure *alone* wasn't enough. Testing against the API in isolation, the model would still sometimes call the tool again; adding that instruction, with the structure already right, was what stopped it. Both were needed, neither alone did it.

The `explain` block recovers something the loop had been throwing away. Left to itself Groq returns a bare tool call — the message carries a `role` and `tool_calls` and **no `content` key at all**. Asked, it fills content in alongside the call:

> *"I'm picking the Tokyo timezone because it's a major city in Japan and a significant financial and cultural hub. To get the current time, I'll use the function to retrieve the current date and time in the Tokyo timezone."*

`Think` keeps that as `decision["reasoning"]`, and `_build_messages` replays it, so later turns see *why* a call was made rather than only that it was.

Its second sentence is a guard, and it is not optional. Asked to explain itself before tool calls, the model starts explaining itself in **final answers** too, returning "the function call was necessary to get the conversion" where the number belongs.

Even guarded, the habit is catching enough that `explain` is a per-instance switch rather than something always on. Notebook 5 turns it off for sub-agents: a leaf worker whose only job is to return a value should be asked for the value, and prompting it to narrate makes it answer its goal with an account of what it did instead. Here, where `Think` is the thing making decisions, it stays on.

`reasoning` also remains optional in the data. `m.get("reasoning")` is `None` when the model didn't offer one, and `content: None` is exactly the shape the API expects for a pure tool call.

**A caution about measuring any of this.** Groq's reliability swings hard from run to run. Measured at four samples, this instruction first looked like a catastrophic regression — four failures out of four. Re-run with a *control*, the unchanged code failed two out of four in the same window, while the instruction produced a valid call with a rationale three times out of three. The first result was noise read as signal. Nothing here is worth concluding from a handful of runs without a control beside it.

The rest of `run()` handles two failure modes chapter 1's notebook 6 will look familiar from:

```python
def run(self, conversation):
    base = self._build_messages(conversation)
    messages = list(base)
    for attempt in range(self.retries):
        try:
            response = client.chat.completions.create(
                model=self.model, messages=messages, tools=all_tools, **self._sampling_kwargs())
        except BadRequestError as e:
            messages = [*base, {"role": "user", "content": tool_call_failure(e) or "..."}]
            continue

        message = response.choices[0].message
        if not message.tool_calls:
            if not message.content:
                messages.append({"role": "assistant", "content": ""})
                messages.append({"role": "user", "content": "Write out your final answer as text."})
                continue
            return {"role": "decision", "type": "respond", "content": message.content}

        call = message.tool_calls[0]
        decision = {"role": "decision", "type": "tool_call",
                    "tool": call.function.name, "content": call.function.arguments}
        if message.content:
            decision["reasoning"] = message.content.strip()
        return decision
    raise ValueError(f"Failed to get a valid decision after {self.retries} attempts")
```

`BadRequestError` is chapter 1's API-level rejection branch — Groq really does reject malformed tool-call generations this way, often enough to matter. But the retry is no longer a shrug. `tool_call_failure()` (chapter 1's `llm.py`) unpacks what Groq actually sent, because a 400 here carries far more than the exception's text suggests:

```
Your last tool call was rejected: tool call validation failed: parameters for tool
delegate did not match schema: errors: [`/broadcast`: expected boolean, but got string]

This is exactly what you generated, and it is not valid:
<function=delegate>{"broadcast": "false", "persistent": "false"}

Issue the call again as one complete, valid tool call. Close every brace and bracket,
give booleans as true/false rather than the strings "true"/"false", ...
```

The model gets its own broken output back, plus the precise reason it was rejected. Same principle as `DebugThink` in notebook 3, which is handed the real schema and the real validation error rather than a nudge — a model told only "that didn't work" re-reads its context, finds nothing new, and makes the same mistake again.

`messages` is rebuilt from `base` on each retry rather than appended to, so the model sees one clear correction instead of a growing stack of past failures.

Worth being precise about what all of this fixes. None of it makes the model smarter about *what* to do. It's about making "already done" legible, and "that was wrong" specific.

## ReAct as a declared loop

Everything is built. Declared as data, ReAct is four components and an index:

In [3]:
runtime = Runtime(
    loop=[
        Observe(event="What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"),
        Think(tools=tools, allow_fork=False),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
        Observe(),
    ],
    repeat_from=1,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    print(f"- {m['role']:9} {m['content']!r}")
    if m.get("reasoning"):
        print(f"             why: {m['reasoning']}")

- user      "What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"
- decision  '{"timezone":"Asia/Tokyo"}'
             why: To get the current time in Tokyo, I need to call the function to get the current time.
- tool      'The current time in Tokyo is 20:53.'
- decision  '{"from_unit":"F","to_unit":"C","value":100}'
             why: I need to convert 100 degrees Fahrenheit to Celsius.
- tool      '100 degrees Fahrenheit is equal to 37.8 degrees Celsius.'
- decision  'The current time in Tokyo is 20:53, and 100 degrees Fahrenheit is 37.78 degrees Celsius.'
- assistant 'The current time in Tokyo is 20:53, and 100 degrees Fahrenheit is 37.78 degrees Celsius.'


`repeat_from=1` is doing all the structural work: position 0 (the user's request) runs once, then `Think → Act → Observe` cycles for as long as it takes. Nothing declared how many tool calls this would need — two, as it happens — and nothing had to. The run ends because `Think` eventually decided to respond rather than call another tool, `Act` turned that decision into an `assistant` message, and `Runtime` saw the role and returned.

(`allow_fork=False` switches off a capability that doesn't exist yet — notebook 5 builds it, and until then every `Think` in this chapter passes it.)

These `Observe()` calls have no extractor, so each result was phrased by the fallback LLM call. That's why the transcript reads in sentences rather than `'2026-08-07 06:54 JST'` and `'37.8'`. The raw values are still available, in `conversation.step_outputs`, exactly as the tools returned them:

In [4]:
print("step_outputs:", conversation.step_outputs)
print("pending:     ", conversation.pending)

step_outputs: ['2026-08-10 20:53 JST', '37.8']
pending:      None


`pending` is back to `None` — the last `Observe` consumed it, as it should. This is the same ReAct behavior chapter 1's `asend_with_tools()` produced, reached through components that can now be re-ordered into something else. Notebook 4 does exactly that, and the runtime doesn't change.

## A loop that can't stop

`repeat_from` bought a real loop, and with it a real hazard: the only exit built so far is a `Think` that voluntarily decides to respond. If it never does, `run()` never returns.

That isn't hypothetical. A tool that keeps failing, a request that can't be satisfied with the tools available, a model that gets stuck rephrasing the same call — all of them produce a loop that spins, spending money on every turn. Chapter 1 never faced this because `for _ in range(retries)` couldn't run more than three times by construction. Trading that ceiling for `repeat_from` means putting a real one back.

`Runtime` takes four independent budgets, any of which can be left as `None`:

```python
def _stop_reason(self, conversation):
    if self.cancel is not None and self.cancel():
        return "cancelled"
    if self.max_iterations is not None and self._iterations >= self.max_iterations:
        return f"reached the maximum of {self.max_iterations} iterations"
    if self.max_tool_calls is not None and self._tool_calls_made(conversation) >= self.max_tool_calls:
        return f"reached the maximum of {self.max_tool_calls} tool calls"
    if self.max_tokens is not None:
        used = sum(count_tokens(m["content"]) for m in conversation.messages if m.get("content"))
        if used >= self.max_tokens:
            return f"reached the maximum token budget of {self.max_tokens}"
    if self.max_seconds is not None and time.monotonic() - self._started_at >= self.max_seconds:
        return f"reached the maximum execution time of {self.max_seconds}s"
    return self._doom_loop(conversation)
```

They measure genuinely different things and none substitutes for another: `max_iterations` counts components run, `max_tool_calls` counts side effects on the outside world, `max_tokens` caps spend, `max_seconds` caps wall-clock latency. `cancel` is a callable checked every iteration — the hook for a user pressing stop.

`count_tokens()` is chapter 1's, from notebook 3, used differently here. There it decided what to *drop* from history to fit a context window; here it decides when to *stop* the whole run. Same measurement, opposite purpose.

### Detecting a loop that's stuck before the budget runs out

Budgets always fire eventually, but "eventually" can be expensive. A stuck loop is usually recognizable long before it exhausts anything, and it looks like one of four shapes:

```python
def _doom_loop(self, conversation, window=4):
    decisions = [m for m in conversation.messages if m.get("role") == "decision" and m.get("type") == "tool_call"]
    recent = decisions[-window:]

    if len(recent) >= 2 and len({(d["tool"], d["content"]) for d in recent[-2:]}) == 1:
        return "the same tool call with identical arguments repeated"

    errors = [m["content"] for m in conversation.messages if m.get("role") == "tool" and m["content"].startswith("Error")]
    if len(errors) >= 2 and errors[-1] == errors[-2]:
        return "the same error repeated"

    if len(recent) == window:
        pairs = [(d["tool"], d["content"]) for d in recent]
        if pairs[0] == pairs[2] and pairs[1] == pairs[3] and pairs[0] != pairs[1]:
            return "oscillating between the same two calls"

    answered = [d for d in recent if "result" in d]
    if len(answered) >= 2:
        before, last = answered[-2:]
        if (before["tool"] == last["tool"] and before["result"] == last["result"]
                and _similar_args(before["content"], last["content"])):
            return f"`{last['tool']}` was asked the same thing again and returned the same result"

    return None
```

An identical repeat, an identical error twice running, an A-B-A-B oscillation — and the one a model actually does most, which the first three all miss: **the same question in slightly different words, getting the same answer back.** `04_memory`'s recall agent searched for *"checkout slowed 6th of November"*, then for *"checkout slowed **on** 6th of November"*, and got the same listing both times. The arguments differ, so the identical-repeat check never fires.

The fourth check needs three things to agree, not one. Same result alone would be wrong: converting −40 °F to Celsius and −40 °C to Fahrenheit both return `-40.0`, and those are two different questions. So it also asks whether the arguments are nearly the same text — compared on their *values*, since the keys are identical in any two calls to one tool. And it compares what the tool really returned, which `Act` keeps on the call it carried out as `decision["result"]`, not `Observe`'s one-sentence rewrite, which varies run to run.

All four are comparisons between values already sitting in `messages` — set equality and tuple equality, no model call, no judgment about whether progress is being made. That restraint matters: a component that *judged* progress would be reasoning, and reasoning belongs to `Think`.

Getting a live model to reproduce all three on demand isn't reliable — which is the point, they're failure modes, not features. So seed each shape by hand and check the detector directly, the same way chapter 1's notebook 9 hand-seeded a validation failure rather than waiting for one:

In [3]:
def seeded(decisions, tool_results=()):
    c = Conversation()
    for tool, args, *result in decisions:
        # a third element is what the tool returned -- Act keeps it on the call it carried out
        c.messages.append({"role": "decision", "type": "tool_call", "tool": tool, "content": args,
                           **({"result": result[0]} if result else {})})
    for result in tool_results:
        c.messages.append({"role": "tool", "content": result})
    return c

probe = Runtime()

same_call = seeded([("get_current_time", '{"timezone":"Asia/Tokyo"}')] * 2)
oscillating = seeded([("get_current_time", '{"timezone":"Asia/Tokyo"}'),
                      ("convert_temperature", '{"value":100,"from_unit":"F","to_unit":"C"}')] * 2)
repeated_error = seeded([], ["Error: 'Mars/Olympus' is not a valid IANA timezone"] * 2)
healthy = seeded([("get_current_time", '{"timezone":"Asia/Tokyo"}'),
                  ("convert_temperature", '{"value":100,"from_unit":"F","to_unit":"C"}')])

LISTING = "#1  2026-11-06  Backfill job locking orders table caused checkout slowdown"
reworded = seeded([("search", '{"query":"checkout slowed 6th of November"}', LISTING),
                   ("search", '{"query":"checkout slowed on the 6th of November"}', LISTING)])
# a different question that happens to have the same answer: -40 is the same in both scales
coincidence = seeded([("convert_temperature", '{"value":-40,"from_unit":"F","to_unit":"C"}', "-40.0"),
                      ("convert_temperature", '{"value":-40,"from_unit":"C","to_unit":"F"}', "-40.0")])

for name, c in [("same call twice", same_call), ("A-B-A-B", oscillating),
                ("same error twice", repeated_error), ("reworded, same result", reworded),
                ("two different calls", healthy), ("-40 both ways", coincidence)]:
    print(f"{name:22} -> {probe._doom_loop(c)}")

same call twice        -> the same tool call with identical arguments repeated
A-B-A-B                -> oscillating between the same two calls
same error twice       -> the same error repeated
reworded, same result  -> `search` was asked the same thing again and returned the same result
two different calls    -> None
-40 both ways          -> None


Four caught, and — just as important — both healthy conversations return `None`. A detector that fired on two different tool calls would break every real agent on its second turn, and one that fired on −40 both ways would cut short a question that had not finished.

### Stopping, and saying so

When a stop condition does fire, the run can't just return: whoever called it is waiting for an answer, and returning a conversation whose last message is a half-finished tool result gives them nothing they can use.

The first version of this appended *"Stopping: {reason}. Last step: … This is partial progress, not a final answer."* That is honest, and it throws away everything the tools found — even when the answer was already sitting in the last result and the model had simply not got round to saying it. So a stopped run now gets one more turn:

```python
def _end_gracefully(self, conversation, reason, abandoned=()):
    conversation.stopped = reason
    ...
    finisher = _finisher(self.loop)                 # the last Think in the loop
    gathered = any(m.get("role") == "tool" or "result" in m for m in conversation.messages)
    if reason != "cancelled" and finisher is not None and gathered:
        answer = finisher.finish(conversation, reason)
        if answer:
            conversation.messages.append({"role": "assistant", "content": answer, "stopped": reason})
            return conversation
    ...                                             # otherwise the "Stopping: ..." message, as before
```

`Think.finish` sends the same history with **no tools at all**, so answering is the only move left, and ends it with a note: *you have been stopped; reply using only facts that appear in the tool results; for any part they do not answer, say it was not found.* Every word of that was measured against the real model, and the first wordings failed badly:

- **Put first, as a system message**, the model ignored it and tried to search again — three runs of three.
- **Asked to "answer the request now"**, it answered three of three — with a batch size and a recovery time that appear nowhere in anything it had been shown. Asked for an answer, it made one up.
- **As it stands**, six of six were right: *not found* when the results held only a listing, the exact figures once they held the document.

An invented answer is worse than no answer, which is why the wording is that strict.

Three cases keep the old message. **Cancelled** — `cancel` is somebody saying *stop now*, and they do not want another model call. **Nothing gathered** — there is nothing to answer from. And **no reply** — if the last turn comes back empty, the honest stop is still there to fall back on.

Either way the stop is recorded as `conversation.stopped`, and on the final message as `"stopped"`. That matters more than it looks: a stopped run now ends on something that reads like a real answer, so its text can no longer tell you it was cut short. Chapter 6 found this out the hard way — a supervisor that took a specialist's *"Stopping: …"* for an answer — and now reads `stopped` instead.

Live, with a cap too small for the task:

In [4]:
capped = Runtime(
    loop=[
        Observe(event="What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"),
        Think(tools=tools, allow_fork=False),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=4,
)

stopped = capped.run(Conversation())
for m in stopped.messages:
    print(f"- {m['role']:9} {m['content']!r}")
print()
print("stopped:", stopped.stopped)

- user      "What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"
- decision  '{"timezone":"Asia/Tokyo"}'
- tool      'The current time in Tokyo is 2026-09-28 02:52 JST.'
- assistant 'The current time in Tokyo is 2026-09-28\u202f02:52\u202fJST.  \nThe conversion of 100\u202f°F to °C was not found.'

stopped: reached the maximum of 4 iterations


Four iterations, then a stop — mid-task. The Tokyo half had come back and the conversion had never run, and the last message says exactly that: the time, from the result, and *not found* for the rest, rather than a guess at 37.8. Nothing in the text says the run was cut off; `stopped` does.

One last piece, which does nothing visible yet. Each iteration, `Runtime` writes what's left of every budget onto the conversation:

```python
conversation.remaining_budget = self._remaining_budget(conversation)
```

`Think` reads it. A `Think` holding tools is told, in its system message, roughly how much room is left — *"Budget left before this run is stopped: about 5 turns"* — in turns, not iterations, because iterations count components: `max_iterations=16` on a Think/Act/Observe loop is about five model turns, which is not what anyone reading the number guesses. Without it the model has no idea a budget exists until the runtime stops it, and cannot tell that a sixth search is one it cannot afford.

It matters again in notebook 5, where a `Think` that splits work across sub-agents has to divide a shared budget between them — and can only do that if it knows what's left.

## What you built

✓ Built `Runtime` — a loop that walks a declared list, with `next()` dispatching on an integer and never on content, and `repeat_from` turning a one-way pipeline into a real cycle

✓ Built `Act`, executing an already-made decision and reusing chapter 1's `execute_tool_call()` rather than reimplementing tool execution, and `Observe`, converting a raw effect into a readable observation via either a deterministic extractor or an LLM fallback

✓ Kept raw tool output out of `messages` entirely — `pending` as a one-step handoff, `step_outputs` as an indexed record — so a large result costs context once instead of on every future turn

✓ Built `Think`, with `_build_messages()` reconstructing the `tool_calls`/`tool_call_id` structure that marks work as done — the structure flattening destroys, and without which the model re-requests tools it has already run — plus the system instruction that turned out to be needed alongside it

✓ Captured the model's own rationale for each tool call, which Groq withholds unless asked for, so a `decision` records *why* and not just *what* — and replayed it on later turns

✓ Made a rejected tool call recoverable by handing back Groq's actual complaint — the malformed text it produced and the field that failed validation — rather than a generic "try again" the model can't act on

✓ Ran ReAct as a declared loop through `Runtime`, reaching chapter 1's behavior with components that can now be re-ordered

✓ Made the loop safe to leave running: four independent budgets, four doom-loop shapes verified against hand-seeded conversations (and verified not to fire on healthy ones, including a coincidence), and a stop that answers from what was gathered — only from what was gathered — and records that it stopped

**Next:** `03_retries_and_reflection.ipynb` — two ways the loop notices something is wrong and goes again: repairing bad tool arguments, and critiquing an answer that isn't good enough yet.